In [ ]:
import re
from datetime import datetime
from pathlib import Path

import duckdb

print(f"the code is running at time: {datetime.now()}")

repo_root = (
    Path(__file__).resolve().parents[2] if "__file__" in globals() else Path.cwd().parents[2]
)
CHUNKS = repo_root / "data" / "chunks" / "sentence_bound"
DB = repo_root / "data" / "duckdb" / "modern_wisdom.duckdb"
print(f"CHUNKS: {CHUNKS}")
print(f"DB: {DB}")
assert DB.exists(), f"Missing {DB}"
assert CHUNKS.exists(), f"Missing {CHUNKS}"

con = duckdb.connect(DB.as_posix(), read_only=True)

# 1) Inspect episodes table and dates
schemas = [
    r[0]
    for r in con.execute(
        "select table_schema from information_schema.tables where table_name='episodes'"
    ).fetchall()
]
print("episode tables in schemas:", schemas)

tbl = None
for sch in ("mw", "mw_staging", "main"):
    if sch in schemas:
        tbl = f"{sch}.episodes"
        break
assert tbl, "no episodes table found"

print(con.execute(f"select count(*) from {tbl}").fetchone())
print(con.execute(f"select min(publish_date), max(publish_date) from {tbl}").fetchone())

# 2) Find episodes by metadata tokens AND year
q = "Chris views on discipline 2021 2024"
years = [int(y) for y in re.findall(r"\b(20\d{2})\b", q)]
cols = {
    r[0]
    for r in con.execute(
        "select column_name from information_schema.columns where table_name='episodes' and table_schema=?",
        [tbl.split(".")[0]],
    ).fetchall()
}
date_col = "publish_date" if "publish_date" in cols else None
search_cols = [c for c in ("title", "guest", "headline", "description", "summary") if c in cols]

likes = " OR ".join([f"{c} ILIKE ?" for c in search_cols]) if search_cols else "TRUE"
params = [f"%discipline%"] * len(search_cols)

where = "where (" + likes + ")"
if date_col and years:
    where += (
        f" and extract(year from try_cast({date_col} as DATE)) in ({','.join(map(str, years))})"
    )

sql = f"""
select id, {date_col if date_col else "NULL"} as d, {", ".join(search_cols) if search_cols else "'' as title"}
from {tbl}
{where}
order by {date_col} desc nulls last
limit 10
"""
print(sql)
print(con.execute(sql, params).fetchdf())

# 3) Tokenized chunk scan (AND of tokens) with union_by_name and dynamic text column
tokens = [
    t
    for t in re.findall(r"[A-Za-z0-9']+", q.lower())
    if t not in {"the", "and", "or", "vs", "on", "views", "view"} and len(t) >= 3
]
if not tokens:
    tokens = ["discipline"]
args = [f"%{t}%" for t in tokens]
glob = (CHUNKS / "episode_id=*" / "part-*.parquet").as_posix()

# Discover columns from the parquet union (LIMIT 0 returns schema)
cols_df = con.execute(
    "SELECT * FROM read_parquet(?, hive_partitioning=1, union_by_name=1) LIMIT 0",
    [glob],
).fetchdf()
cols = set(cols_df.columns)

text_candidates = [
    "text",
    "content",
    "segment_text",
    "segment",
    "transcript",
    "raw_text",
    "body",
    "snippet",
    "utterance",
    "line",
    "value",
]
text_col = next((c for c in text_candidates if c in cols), None)
print(f"sorted_cols: {sorted(cols)}")
if text_col is None:
    raise RuntimeError(f"No text-like column found in chunks; saw columns: {sorted(cols)}")

pred = " AND ".join([f"lower({text_col}) like ?"] * len(tokens))

sql_hits = f"""
with hits as (
  select coalesce(episode_id,'') as episode_id, count(*) as hits
  from read_parquet('{glob}', hive_partitioning=1, union_by_name=1)
  where {pred}
  group by 1
)
select * from hits order by hits desc limit 10
"""
print("sql_hits", sql_hits)
df_hits = con.execute(sql_hits, args).fetchdf()
print("sql_hits result", df_hits)

# 4) Peek chunks for the top episode
if not df_hits.empty:
    eid = df_hits.iloc[0]["episode_id"]
    sql_chunks = f"""
      select episode_id, start_ts, end_ts,
             left({text_col}, 160) as snippet
      from read_parquet('{glob}', hive_partitioning=1, union_by_name=1)
      where episode_id=? and {pred}
      order by start_ts
      limit 6
    """
    print(con.execute(sql_chunks, [eid, *args]).fetchdf())
else:
    print("No chunk hits. Try reducing tokens to ['discipline'] or verify chunk path.")


the code is running at time: 2025-10-19 16:24:15.754750
CHUNKS: /Users/ettyekhon/code/src/petroineos/petroineos-related/modern-wisdom-llm-native-pipeline/data/chunks/sentence_bound
DB: /Users/ettyekhon/code/src/petroineos/petroineos-related/modern-wisdom-llm-native-pipeline/data/duckdb/modern_wisdom.duckdb
episode tables in schemas: ['mw', 'mw_staging']
(998,)
('2018-02-12', '2025-09-29')

select id, publish_date as d, title, guest, headline, description
from mw.episodes
where (title ILIKE ? OR guest ILIKE ? OR headline ILIKE ? OR description ILIKE ?) and extract(year from try_cast(publish_date as DATE)) in (2021,2024)
order by publish_date desc nulls last
limit 10

                                     id           d  \
0  e04d8930-ac49-11ee-9f02-9bda25edbfb4  2024-11-18   
1  df723e02-ac49-11ee-9f02-978dd878f43b  2024-10-14   
2  dacfb7a0-c74c-4dbe-899f-e4b55133e954  2021-03-27   

                                               title             guest  \
0  #866 - Jesse James West - H

In [9]:
glob = (CHUNKS / "episode_id=*" / "part-*.parquet").as_posix()
sql = f"""
with hits as (
  select coalesce(episode_id,'') as episode_id, count(*) as hits
  from read_parquet('{glob}', hive_partitioning=1, union_by_name=1)
  where lower(text) like '%discipline%'
  group by 1
)
select * from hits order by hits desc limit 10
"""
print(con.execute(sql).fetchdf())


                             episode_id  hits
0  ff70da5a-63c3-40dc-b96f-8d922d098bce    25
1  6555ee82-fa53-11ed-be47-af3569368d6f    13
2  05ddde4d-987e-4674-b6af-c4d57273f910    12
3  7360d681-77bc-40fd-9fca-3a3da8712c1e    12
4  af180a60-bc1b-11ef-b1b1-537db0770e70    12
5  cf7b7c72-0eb3-11ee-8da3-87c1e3ad7b07    11
6  23a8348e-bc14-11ef-8994-37093be6cfe6    10
7  dd359436-ac49-11ee-9f02-dfcc457d9ca8     7
8  d236f1e4-0eb3-11ee-8da3-5303d3b9178c     7
9  d4df3b34-ac49-11ee-9f02-4f628d202ae8     7
